# MinerU trên Google Colab - Hỗ trợ tiếng Việt

Script chạy MinerU (fork với hỗ trợ tiếng Việt) trên Google Colab.

**Fork:** https://github.com/spkzr/MinerU

## 1. Cài đặt môi trường

In [ ]:
# Chạy cell này trước - Cài đặt MinerU + Tesseract (cho OCR tiếng Việt)
# Tesseract: dùng cho nhận dạng tiếng Việt thay PaddleOCR
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie
!pip install -q pytesseract "git+https://github.com/spkzr/MinerU.git#egg=mineru[all]"

In [ ]:
!mineru --version

## 2. Upload PDF

In [ ]:
!mkdir -p /content/input /content/output

In [ ]:
# Bấm "Choose Files" để upload PDF
from google.colab import files
uploaded = files.upload()

import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"/content/input/{filename}")
    print(f"Đã upload: {filename}")

## 3. Chạy MinerU với tiếng Việt

In [ ]:
import os
import subprocess

input_files = [f for f in os.listdir("/content/input") if f.endswith(".pdf")]
print("Các file PDF:", input_files)

if not input_files:
    raise FileNotFoundError("Chưa có file PDF. Hãy upload ở cell trên.")

PDF_FILE = f"/content/input/{input_files[0]}"
print(f"Đang xử lý: {PDF_FILE}")

result = subprocess.run(
    ["mineru", "-p", PDF_FILE, "-o", "/content/output", "-l", "vi", "-b", "pipeline"],
    capture_output=True,
    text=True,
    cwd="/content"
)
if result.stdout:
    print(result.stdout)
if result.returncode == 0:
    print("Hoàn thành!")
else:
    print("Lỗi:", result.returncode)
    if result.stderr:
        print("Chi tiết:", result.stderr[-2000:])

## 4. Xem và tải kết quả

In [ ]:
import glob
md_files = glob.glob("/content/output/**/*.md", recursive=True)
print("Các file Markdown:", md_files)

if md_files:
    with open(md_files[0], "r", encoding="utf-8") as f:
        print(f.read()[:3000])

In [ ]:
# Tải toàn bộ thư mục output (Markdown + hình ảnh + bảng) dưới dạng zip
import shutil
import os

out_dir = "/content/output"
zip_path = "/content/mineru_output.zip"
if os.path.exists(out_dir) and os.listdir(out_dir):
    shutil.make_archive(zip_path.replace(".zip", ""), "zip", out_dir)
    from google.colab import files
    files.download(zip_path)
    print("Đã tải mineru_output.zip!")
else:
    print("Chưa có kết quả để tải.")

In [ ]:
# Tải file Markdown về máy
import glob
md_files = glob.glob("/content/output/**/*.md", recursive=True)
if md_files:
    from google.colab import files
    files.download(md_files[0])
    print("Đã tải file!")
else:
    print("Chưa tìm thấy file .md")